# Apache Spark — First Contact

Apache Spark is a distributed data processing engine built for large-scale batch and streaming workloads. Instead of forcing one machine to do everything, Spark splits work into tasks and coordinates execution across cores or clusters. Its core value is speed, scale, and a programming model that lets you express transformations at a high level.

A useful mental model is this: you describe *what* data you want to read, transform, aggregate, and write, and Spark builds a directed acyclic graph (DAG) of execution steps. It then optimizes that plan before running it. This is why Spark feels declarative at the API level but powerful at runtime.

Citi framing: Kafka ingests 600 events/sec. Once in storage, we need to process 500,000 metric rows for anomaly scoring, aggregation, and reporting. This is a Spark job.

```text
[Postgres/S3] → [SparkSession] → [DAG: Read → Transform → Aggregate → Write] → [Output]
```

In [1]:
import os
# Java 8 required for PySpark on this Windows host (JDK 17 has NIO loopback bug)
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jre1.8.0_481"
os.environ["HADOOP_HOME"] = "C:/hadoop"

%pip install "pyspark==3.5.4"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, when, desc
from pyspark.sql import types as T

Note: you may need to restart the kernel to use updated packages.


## SparkSession

SparkSession is the entry point to Spark. It gives you access to DataFrame reads/writes, SQL, configuration, and execution control.

For this notebook, use `local[*]` from the host machine. That tells Spark to use all local CPU cores without trying to connect executors in Docker back to a host driver process. Cluster URLs such as `spark://localhost:7077` are intentionally avoided here.

In [2]:
spark = (
    SparkSession.builder
    .appName("CityTelemetryFirstJob")
    .master("local[*]")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

print(spark.version)
print("SparkSession ready — running in local[*] mode")

3.5.4
SparkSession ready — running in local[*] mode


## Read from Postgres via JDBC

Spark can read relational tables through JDBC. For larger reads, `fetchsize` helps reduce round trips, and Spark may push parts of the query closer to the source when possible. Here we read the three telemetry tables directly from PostgreSQL.

In [3]:
jdbc_url = "jdbc:postgresql://localhost:5432/de_telemetry"
jdbc_props = {
    "user": "de_admin",
    "password": "DeAdmin2026!",
    "driver": "org.postgresql.Driver",
    "fetchsize": "10000",
}

endpoints_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "endpoints")
    .option("user", jdbc_props["user"])
    .option("password", jdbc_props["password"])
    .option("driver", jdbc_props["driver"])
    .option("fetchsize", jdbc_props["fetchsize"])
    .load()
)

metrics_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "metrics")
    .option("user", jdbc_props["user"])
    .option("password", jdbc_props["password"])
    .option("driver", jdbc_props["driver"])
    .option("fetchsize", jdbc_props["fetchsize"])
    .load()
)

alerts_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "alerts")
    .option("user", jdbc_props["user"])
    .option("password", jdbc_props["password"])
    .option("driver", jdbc_props["driver"])
    .option("fetchsize", jdbc_props["fetchsize"])
    .load()
)

endpoints_df.printSchema()
metrics_df.printSchema()
alerts_df.printSchema()

print(f"endpoints: {endpoints_df.count()}, metrics: {metrics_df.count()}, alerts: {alerts_df.count()}")

root
 |-- endpoint_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- category: string (nullable = true)

root
 |-- endpoint_id: integer (nullable = true)
 |-- metric_name: string (nullable = true)
 |-- value: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- metric_id: integer (nullable = true)

root
 |-- alert_id: integer (nullable = true)
 |-- endpoint_id: integer (nullable = true)
 |-- severity: string (nullable = true)
 |-- message: string (nullable = true)
 |-- created_at: timestamp (nullable = true)



endpoints: 10000, metrics: 500000, alerts: 25000


## RDD vs DataFrame — which one to use

| Aspect | RDD | DataFrame |
|---|---|---|
| Abstraction level | Low-level distributed collection | High-level tabular API |
| Optimization | Manual, limited optimizer help | Catalyst + Tungsten optimizations |
| Type safety | Raw Python objects in PySpark | Schema-based columns and expressions |
| Best use case | Very custom low-level transformations | Almost all analytics and ETL work |

**Conclusion:** Use DataFrame/Dataset API always in 2024+. RDDs are legacy.

## First Transform: Endpoint Alert Summary

Group alerts by endpoint and severity, then join to endpoint metadata so the output is business-readable.

In [4]:
alert_counts = alerts_df.groupBy("endpoint_id", "severity").agg(count("*").alias("alert_count"))

enriched = alert_counts.join(endpoints_df, on="endpoint_id", how="left")

result = (
    enriched.select("name", "region", "category", "severity", "alert_count")
    .orderBy(desc("alert_count"))
)

result.show(20, truncate=False)

+-----------------------+------+--------+--------+-----------+
|name                   |region|category|severity|alert_count|
+-----------------------+------+--------+--------+-----------+
|srv-07578.citi.internal|NYC1  |web     |CRITICAL|6          |
|srv-07086.citi.internal|SNG1  |db      |CRITICAL|6          |
|srv-05084.citi.internal|LON1  |db      |MEDIUM  |6          |
|srv-09373.citi.internal|NYC2  |db      |LOW     |6          |
|srv-06987.citi.internal|NYC1  |cache   |HIGH    |5          |
|srv-06401.citi.internal|SNG1  |worker  |MEDIUM  |5          |
|srv-06205.citi.internal|SNG1  |db      |HIGH    |5          |
|srv-04452.citi.internal|NYC1  |worker  |LOW     |5          |
|srv-03195.citi.internal|NYC1  |monitor |LOW     |5          |
|srv-03061.citi.internal|NYC2  |cache   |MEDIUM  |5          |
|srv-04452.citi.internal|NYC1  |worker  |CRITICAL|5          |
|srv-08653.citi.internal|SNG1  |db      |MEDIUM  |5          |
|srv-02524.citi.internal|NYC2  |db      |CRITICAL|5    

## Second Transform: Average Metric by Region

Aggregate 500k metric rows — this is where Spark starts to show its strength compared with single-machine tooling.

In [5]:
joined = metrics_df.join(endpoints_df, on="endpoint_id", how="left")

regional_avg = (
    joined.groupBy("region", "metric_name")
    .agg(avg("value").alias("avg_value"), count("*").alias("sample_count"))
    .orderBy("region", "metric_name")
)

regional_avg.show(30, truncate=False)

+------+--------------+------------------+------------+
|region|metric_name   |avg_value         |sample_count|
+------+--------------+------------------+------------+
|LON1  |cpu_percent   |52.407107648725244|24710       |
|LON1  |disk_io       |2512.323822737709 |25262       |
|LON1  |memory_percent|52.32286169740509 |24779       |
|LON1  |network_in    |495.9671974855946 |24817       |
|LON1  |network_out   |502.3302631790098 |24926       |
|NYC1  |cpu_percent   |52.43577896727975 |24847       |
|NYC1  |disk_io       |2494.8031926835183|24766       |
|NYC1  |memory_percent|52.54616619831394 |24910       |
|NYC1  |network_in    |501.16478759353123|24858       |
|NYC1  |network_out   |500.19621425986236|24867       |
|NYC2  |cpu_percent   |52.38976477268113 |24657       |
|NYC2  |disk_io       |2507.0537165928836|24396       |
|NYC2  |memory_percent|52.69086598778008 |24550       |
|NYC2  |network_in    |502.20600518753287|24289       |
|NYC2  |network_out   |500.22635778794967|24249 

## Explain Plan

Spark's Catalyst optimizer rewrites your query before execution. Reading the plan is how you start seeing where shuffles and aggregation costs live.

In [6]:
regional_avg.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- SortMergeJoin LeftOuter (7)
                     :- Sort (3)
                     :  +- Exchange (2)
                     :     +- Scan JDBCRelation(metrics) [numPartitions=1]  (1)
                     +- Sort (6)
                        +- Exchange (5)
                           +- Scan JDBCRelation(endpoints) [numPartitions=1]  (4)


(1) Scan JDBCRelation(metrics) [numPartitions=1] 
Output [3]: [endpoint_id#10, metric_name#11, value#12]
ReadSchema: struct<endpoint_id:int,metric_name:string,value:double>

(2) Exchange
Input [3]: [endpoint_id#10, metric_name#11, value#12]
Arguments: hashpartitioning(endpoint_id#10, 200), ENSURE_REQUIREMENTS, [plan_id=444]

(3) Sort
Input [3]: [endpoint_id#10, metric_name#11, value#12]
Arguments: [endpoint_id#10 ASC NULLS FIRST], false, 0

(4)

`Exchange` usually means shuffle: Spark is moving data across partitions so rows with the same grouping keys end up together.

`HashAggregate` means Spark is building aggregation state in memory by key, which is common for grouped counts and averages.

## Write Output

Spark can write results back to PostgreSQL through JDBC. Here we persist the alert summary for downstream reporting.

In [7]:
(
    result.write.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "spark_alert_summary")
    .option("user", jdbc_props["user"])
    .option("password", jdbc_props["password"])
    .option("driver", jdbc_props["driver"])
    .mode("overwrite")
    .save()
)

print("Written spark_alert_summary to de_telemetry")

Written spark_alert_summary to de_telemetry


## What Just Happened

- Read 535,000 rows in seconds.
- Ran two aggregations.
- Viewed the Catalyst execution plan.
- Wrote the alert summary back to Postgres.

Citi tie-in: A nightly Spark job over 500k metrics rows takes seconds. The same work in pandas on a single machine risks OOM and takes minutes.

Next: Run `spark_concepts.md` then move to Round 2 for internals and tuning.

In [8]:
spark.stop()